In [1]:
!pip install opencv-python numpy scikit-learn joblib

In [5]:
import os
import cv2
import joblib
import numpy as np

from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder

In [4]:
DATASET_PATH = r"C:\Users\tripa\PCA_ANN_FACE_RECOGNITION.ipynb\dataset\dataset\faces"

X = []
labels = []

face_detector = cv2.CascadeClassifier(
    cv2.data.haarcascades +
    "haarcascade_frontalface_default.xml"
)

for person in os.listdir(DATASET_PATH):

    person_folder = os.path.join(
        DATASET_PATH,
        person
    )

    if not os.path.isdir(person_folder):
        continue

    print("Processing:", person)

    for image_name in os.listdir(person_folder):

        image_path = os.path.join(
            person_folder,
            image_name
        )

        image = cv2.imread(image_path)

        if image is None:
            continue

        gray = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2GRAY
        )

        faces = face_detector.detectMultiScale(
            gray,
            scaleFactor=1.3,
            minNeighbors=5
        )

        if len(faces) == 0:
            continue

        x, y, w, h = faces[0]

        face = gray[y:y+h, x:x+w]

        face = cv2.resize(
            face,
            (100, 100)
        )

        X.append(face.flatten())
        labels.append(person)

print("Total Faces Found:", len(X))

Processing: .ipynb_checkpoints
Processing: Aamir
Processing: Ajay
Processing: Akshay
Processing: Alia
Processing: Amitabh
Processing: Deepika
Processing: Disha
Processing: Farhan
Processing: Ileana
Total Faces Found: 358


In [6]:
import os
print(os.getcwd())


C:\Users\tripa\PCA_ANN_FACE_RECOGNITION.ipynb\dataset\dataset


In [7]:
X = np.array(X)

encoder = LabelEncoder()

y_encoded = encoder.fit_transform(labels)

print("Labels Encoded Successfully")
print("Total Classes:", len(encoder.classes_))
print("Classes:", encoder.classes_)

Labels Encoded Successfully
Total Classes: 9
Classes: ['Aamir' 'Ajay' 'Akshay' 'Alia' 'Amitabh' 'Deepika' 'Disha' 'Farhan'
 'Ileana']


In [8]:
model = KNeighborsClassifier(
    n_neighbors=3
)

model.fit(
    X,
    y_encoded
)

print("Model Trained Successfully")

Model Trained Successfully


In [9]:
joblib.dump(
    model,
    "face_model.pkl"
)

joblib.dump(
    encoder,
    "label_encoder.pkl"
)

print("Files Saved Successfully")

Files Saved Successfully


In [10]:
import os

print(os.listdir())

['.ipynb_checkpoints', 'faces', 'face_model.pkl', 'Iris', 'label_encoder.pkl', 'requirements.txt', 'train_model.ipynb', 'Untitled.ipynb', 'Untitled1.ipynb']


In [11]:
import os
print(os.listdir())

['.ipynb_checkpoints', 'faces', 'face_model.pkl', 'Iris', 'label_encoder.pkl', 'requirements.txt', 'train_model.ipynb', 'Untitled.ipynb', 'Untitled1.ipynb']


In [14]:
import cv2
import joblib
import numpy as np

# Load trained model
model = joblib.load("face_model.pkl")
encoder = joblib.load("label_encoder.pkl")

# Haar Cascade Face Detector
face_detector = cv2.CascadeClassifier(
    cv2.data.haarcascades +
    "haarcascade_frontalface_default.xml"
)

# Start Webcam
cap = cv2.VideoCapture(0)

while True:

    ret, frame = cap.read()

    if not ret:
        break

    gray = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2GRAY
    )

    faces = face_detector.detectMultiScale(
        gray,
        1.3,
        5
    )

    for (x, y, w, h) in faces:

        face = gray[y:y+h, x:x+w]

        face = cv2.resize(
            face,
            (100, 100)
        )

        face = face.flatten().reshape(1, -1)

        prediction = model.predict(face)

        name = encoder.inverse_transform(
            prediction
        )[0]

        cv2.rectangle(
            frame,
            (x, y),
            (x+w, y+h),
            (0, 255, 0),
            2
        )

        cv2.putText(
            frame,
            name,
            (x, y-10),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 0),
            2
        )

    cv2.imshow(
        "Face Recognition",
        frame
    )

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [13]:
import cv2

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Camera not opened")
else:
    print("Camera opened successfully")

cap.release()

Camera opened successfully


In [15]:
from sklearn.metrics import accuracy_score

# Predict on training data
y_pred = model.predict(X)

# Calculate accuracy
accuracy = accuracy_score(y_encoded, y_pred)

print("Accuracy:", accuracy * 100, "%")

Accuracy: 72.62569832402235 %
